# `web_search.py`

This notebook is organized into logical, executable sections from the corresponding Python source file.

## Setup

Load optional environment support and configure the search client.

In [ ]:
﻿"""Minimal web search adapter for Tavily."""

import os

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

if load_dotenv:
    load_dotenv()




## Web search

Query Tavily safely and normalize its result records.

In [ ]:
def search_web(query: str, k: int = 5) -> list[dict]:
    if not isinstance(query, str) or not query.strip() or k <= 0:
        return []

    key = os.getenv("WEB_SEARCH_API_KEY")
    endpoint = os.getenv("WEB_SEARCH_API_URL", "https://api.tavily.com/search")
    if not key:
        return []

    try:
        import requests
    except ImportError:
        return []

    try:
        response = requests.post(
            endpoint,
            json={"query": query, "max_results": k, "api_key": key},
            timeout=15,
        )
        response.raise_for_status()
        payload = response.json()
    except (requests.RequestException, ValueError, TypeError):
        return []

    results = payload.get("results", []) if isinstance(payload, dict) else []
    return [
        {
            "title": item.get("title", ""),
            "url": item.get("url", ""),
            "snippet": item.get("content", item.get("snippet", "")),
        }
        for item in results[:k]
    ]
